In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q -U ultralytics

import os
MODEL  = ".../best.pt" #đẩy file best.pt lên gg drive và đúng đường dẫn
FOLDER = ".../test_anh"      # sửa đúng thư mục ảnh của bạn
print("Có mô hình:", os.path.exists(MODEL))
print("Có thư mục ảnh:", os.path.exists(FOLDER), os.listdir(FOLDER)[:10] if os.path.exists(FOLDER) else "")

In [ ]:
from ultralytics import YOLO
from PIL import Image
from IPython.display import display

TAU, LOW = 0.7, 0.25
model = YOLO(MODEL)
files = sorted(f for f in os.listdir(FOLDER) if f.lower().endswith((".jpg", ".jpeg", ".png")))

for fn in files:
    r = model.predict(f"{FOLDER}/{fn}", conf=0.25, imgsz=640, device="cpu", verbose=False)[0]
    dets = [(model.names[int(c)], float(p)) for c, p in zip(r.boxes.cls, r.boxes.conf)]
    sure   = [d for d in dets if d[1] >= TAU]
    unsure = [d for d in dets if LOW <= d[1] < TAU]
    if unsure:
        status = "KHÔNG CHẮC -> không đọc tổng, yêu cầu tách tờ / chụp lại"
    elif not sure:
        status = "KHÔNG THẤY TIỀN"
    else:
        status = f"ĐỌC: {len(sure)} tờ {[n for n,_ in sure]} | tổng {sum(int(n) for n,_ in sure):,} đồng"
    print(f"\n{fn} -> {status}")
    print("  chi tiết:", [(n, round(p, 2)) for n, p in dets])
    display(Image.fromarray(r.plot()[..., ::-1]).resize((480, 360)))